In [ ]:
import asyncio
import websockets
from IPython.display import clear_output
import os
from pathlib import Path

def syllable_boundary_normalize(p):
    retval=""
    with open(p,'r',encoding='utf8') as f:
        rawsb=[line.strip().split() for line in f.readlines()]
    #print(rawsb)
    noraw=[]
    for i in range(len(rawsb)):
        if(len(rawsb[i])==5):
            noraw.append(rawsb[i])
    noraw[-1][1]=rawsb[-1][1]
    #print(noraw)
    for i in range(len(noraw)-1):
        noraw[i][1]=noraw[i+1][0]
    #print(noraw)
    for i in range(len(noraw)):
        if(noraw[i][-1]=='<SILENCE>'):
            retval=retval+noraw[i][0]+" "+noraw[i][1]+" ε\n"
        else:
            retval=retval+noraw[i][0]+" "+noraw[i][1]+" "+noraw[i][-1]+'\n'
    #print(retval)
    return retval

recvdir='/home/cewarman/research/my-app/local/recv_data'
wavedir='{}/wavs'.format(recvdir)
textdir='{}/txts'.format(recvdir)
labdir='{}/labs'.format(recvdir)
tmpdir='{}/tmp'.format(recvdir)
os.system("mkdir -p {} && mkdir -p {} && mkdir -p {} && mkdir -p {} ".format(wavedir,textdir,labdir,tmpdir))

async def handler(ws):
    group=''
    wavid=''
    context=''
    async for message in ws:
        # 注意：message 可能是 str（文字）或 bytes（二進位）
        if isinstance(message, bytes):
            with open("{}/{}.wav".format(wavedir,wavid), "wb") as f:
                f.write(message)
            with open("{}/{}.txt".format(textdir,wavid), "w") as f:
                f.write(context)
            print(group,wavid,context)
            wavlstpath='{}/{}_wavlst.txt'.format(tmpdir,wavid)
            txtlstpath='{}/{}_txtlst.txt'.format(tmpdir,wavid)
            os.system('echo {}/{}.wav > {} '.format(wavedir,wavid,wavlstpath))
            os.system('echo {}/{}.txt > {} '.format(textdir,wavid,txtlstpath))
            os.system('cd /home/cewarman/research/kaldi/CEmix/TAAN && python3 gmm_realign.py {} {} {}'.format(txtlstpath,wavlstpath,labdir))
            #clear_output()
            if Path("{}/{}.lab".format(labdir,wavid)).is_file():
                await ws.send(syllable_boundary_normalize("{}/{}.lab".format(labdir,wavid)))
        else:
            if(message[:5]=='group'):
                group=message[5:]
            elif(message[:5]=='wavid'):
                wavid=message[5:]
            elif(message[:7]=='context'):
                context=message[7:]

async def main():
    async with websockets.serve(handler, "0.0.0.0", 9999):
        print("Listening on ws://0.0.0.0:9999")
        await asyncio.Future()



try:
    asyncio.run(main())
except RuntimeError:
    # 如果已經有 event loop（像在 Jupyter）
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(main())

: 

In [2]:
def syllable_boundary_normalize(p):
    retval=""
    with open(p,'r',encoding='utf8') as f:
        rawsb=[line.strip().split() for line in f.readlines()]
    #print(rawsb)
    noraw=[]
    for i in range(len(rawsb)):
        if(len(rawsb[i])==5):
            noraw.append(rawsb[i])
    noraw[-1][1]=rawsb[-1][1]
    #print(noraw)
    for i in range(len(noraw)-1):
        noraw[i][1]=noraw[i+1][0]
    #print(noraw)
    for i in range(len(noraw)):
        if(noraw[i][-1]=='<SILENCE>'):
            retval=retval+noraw[i][0]+" "+noraw[i][1]+" ε\n"
        else:
            retval=retval+noraw[i][0]+" "+noraw[i][1]+" "+noraw[i][-1]+'\n'
    #print(retval)
    return retval
syllable_boundary_normalize('./recv_data/labs/testing.lab')

'0.000000 0.740000 ε\n0.740000 1.090000 次\n1.090000 1.120000 ε\n1.120000 1.360000 等\n1.360000 1.550000 職\n1.550000 2.010000 員\n2.010000 2.040000 ε\n2.040000 2.210000 都\n2.210000 2.420000 被\n2.420000 2.740000 遣\n2.740000 3.090000 散\n3.090000 3.340000 了\n3.340000 3.990000 ε\n'

In [1]:
import os
recvdir='/home/cewarman/research/my-app/local/recv_data'
wavedir='{}/wavs'.format(recvdir)
textdir='{}/txts'.format(recvdir)
labdir='{}/labs'.format(recvdir)
tmpdir='{}/tmp'.format(recvdir)
group="PC"
wavid="testing-1"
context="次等職員都被遣散了"
wavlstpath='{}/{}_wavlst.txt'.format(tmpdir,wavid)
txtlstpath='{}/{}_txtlst.txt'.format(tmpdir,wavid)
os.system('echo {}/{}.wav > {} '.format(wavedir,wavid,wavlstpath))
os.system('echo {}/{}.txt > {} '.format(textdir,wavid,txtlstpath))
os.system('cd /home/cewarman/research/kaldi/CEmix/BTNL/s5/ && bash realign.sh {} {} {}'.format(txtlstpath,wavlstpath,labdir))


/home/cewarman/research/chores/kaldi_DIY/bin/realign_prepare /home/cewarman/research/my-app/local/recv_data/tmp/testing-1_txtlst.txt /home/cewarman/research/my-app/local/recv_data/tmp/testing-1_wavlst.txt /home/cewarman/research/kaldi/CEmix/BTNL/s5/data_alignment/local/dict/lexiconp.txt /home/cewarman/research/kaldi/CEmix/BTNL/s5/data_alignment/alignment
total_wav_size=1, total_text_size=1
useful_wav_size=1, useful_text_size=1
unpaird_text_number: 0
unpaird_wave_number: 0
fix_data_dir.sh: kept all 1 utterances.
fix_data_dir.sh: old files are kept in /home/cewarman/research/kaldi/CEmix/BTNL/s5/data_alignment/alignment/.backup
speak number = 1 ,thread_num = 1
./steps/make_mfcc_pitch.sh --cmd run.pl --nj 1 /home/cewarman/research/kaldi/CEmix/BTNL/s5/data_alignment/alignment /mnt/slave2/kaldi_mfcc/raw/log /mnt/slave2/kaldi_mfcc/raw
utils/validate_data_dir.sh: WARNING: you have only one speaker.  This probably a bad idea.
   Search for the word 'bold' in http://kaldi-asr.org/doc/data_prep.h

0

In [40]:
import librosa
import numpy as np
y, sr = librosa.load('recv_data/wavs/1_1_Male_1_1_A_fan1_PC_1.wav', sr=16000)
print(len(y),sr)
#f0, voiced_flag, voiced_probs = librosa.pyin(y,sr=sr, fmin=50, fmax=600, fill_na=0, hop_length=160)
#print(len(f0),f0,voiced_probs)
#f0[voiced_probs < 0.0101] = 0
f0=librosa.yin(y, frame_length=2048, fmin=50, fmax=500, hop_length=160, sr=sr)
condition = f0 <= 50 
f0[condition] = 0
condition = f0 >= 500
f0[condition] = 0
print(f0)
str_f0 = " ".join(map(str, np.log(f0+1)))
print(str_f0)
with open('recv_data/wavs/1_1_Male_1_1_A_fan1_PC_1.pitch','w') as f:
    for i in range(len(f0)):
        f.write("{}\n".format(f0[i]))

31360 16000
[  0.         140.72069251 146.02924768 140.06130861 138.52883368
 143.19893119 138.30100359 138.86045673 139.10978473 140.75171038
 141.74499376 138.85181641 139.23946544 142.08323345 141.21364533
 149.49194341 150.24794337 148.82142466 150.40379511 150.27873628
 149.66123564 149.89433764 148.57696555 149.24741681 150.70546338
 151.24327179 151.55093942 151.4238662  151.5449506  151.39661662
 151.24029474 150.44117442 151.09438258 151.4479439   70.32585552
 151.30984362  70.23927834  69.7887535   69.97420689  69.84673484
  69.83777365  69.74105838  69.81734003  69.81990987  69.77988514
  68.92490675  68.62508703 154.61194812 149.51877722 144.67669485
 139.38603662  70.09521078  70.24538806  69.90912998  70.20173394
  70.26682496  70.35473354  70.13920019  69.81450552  69.57176105
  69.37135842  69.74574143 158.74062456  68.9506991  155.50470306
  69.12107808  69.10834909  68.89942989  68.99627001  69.25825699
  69.78115951  70.11215411  70.20705323  70.33402647  69.8649992

In [58]:
import parselmouth
snd = parselmouth.Sound('recv_data/wavs/1_1_Male_1_1_A_ji1tui3_PC_3.wav')
pitch = snd.to_pitch_ac(time_step=0.01, pitch_floor=75, pitch_ceiling=500)
print(pitch, pitch.selected_array['frequency'],type(pitch.selected_array['frequency']))

Object type: Pitch
Object name: <no name>
Date: Thu Jan 22 11:17:32 2026

Time domain:
   Start time: 0 seconds
   End time: 1.896 seconds
   Total duration: 1.896 seconds
Time sampling:
   Number of frames: 186 (47 voiced)
   Time step: 0.01 seconds
   First frame centred at: 0.023000000000000017 seconds
Ceiling at: 500 Hz

Estimated quantiles:
   10% = 179.503761 Hz = 155.345435 Mel = 10.1281689 semitones above 100 Hz = 4.92230405 ERB
   16% = 181.632665 Hz = 156.948157 Mel = 10.3322842 semitones above 100 Hz = 4.96898121 ERB
   50% = 411.105029 Hz = 306.990979 Mel = 24.4740842 semitones above 100 Hz = 9.06211865 ERB
   84% = 484.678039 Hz = 347.560018 Mel = 27.3243206 semitones above 100 Hz = 10.0901244 ERB
   90% = 487.979427 Hz = 349.312131 Mel = 27.4418439 semitones above 100 Hz = 10.1338851 ERB
Estimated spreading:
   84%-median = 74.37 Hz = 41.01 Mel = 2.881 semitones = 1.039 ERB
   median-16% = 232 Hz = 151.7 Mel = 14.29 semitones = 4.137 ERB
   90%-10% = 311.8 Hz = 196.1 Mel 

In [46]:
pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 14.7 MB/s eta 0:00:00 0:00:01m
Note: you may need to restart the kernel to use updated packages.


In [56]:
import pickle
data=pickle.load(open("recv_data/wavs/1_1_Male_1_1_A_fan1_PC_1.pitch",'rb'))


EOFError: Ran out of input